Team coursework case study: a synthetic coffee subscription business. The notebook generates its own simulated data with Faker, builds a SQLite database, and demonstrates SQL analysis. Outputs have been removed; no generated customer records or database are included.


In [ ]:
!pip install faker
import random
import csv
import sqlite3
import pandas as pd
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from google.colab import files
from faker import Faker

fake = Faker('en_GB')

In [ ]:
# Set 1 year of operation.
end_date = datetime(2026, 2, 28)
start_date = end_date - relativedelta(years=1)

In [ ]:
# Database Schema Setup

conn = sqlite3.connect('bean_co.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

In [ ]:
# 1. Create tables
cursor.executescript('''
CREATE TABLE customer (customer_id INTEGER PRIMARY KEY AUTOINCREMENT, first_name TEXT NOT NULL, last_name TEXT NOT NULL, email TEXT UNIQUE NOT NULL, phone TEXT, gender TEXT, date_of_birth TEXT, created_at TEXT);
CREATE TABLE subscription_plan (subscription_plan_id INTEGER PRIMARY KEY AUTOINCREMENT, subscription_plan_name TEXT, price REAL, description TEXT);
CREATE TABLE cancel_reason (reason_id INTEGER PRIMARY KEY AUTOINCREMENT, reason_label TEXT);
CREATE TABLE supplier (supplier_id INTEGER PRIMARY KEY AUTOINCREMENT, company_name TEXT, contact_person TEXT, email TEXT, country_of_origin TEXT);
CREATE TABLE product_category (category_id INTEGER PRIMARY KEY AUTOINCREMENT, category_name TEXT);
CREATE TABLE address (address_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER, address_type TEXT, street_line1 TEXT, street_line2 TEXT, postcode TEXT, city TEXT, country_code TEXT, is_default INTEGER, FOREIGN KEY (customer_id) REFERENCES customer(customer_id));
CREATE TABLE subscription (subscription_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER, subscription_plan_id INTEGER, start_date TEXT, next_billing_date TEXT, status TEXT, auto_renew INTEGER, FOREIGN KEY (customer_id) REFERENCES customer(customer_id), FOREIGN KEY (subscription_plan_id) REFERENCES subscription_plan(subscription_plan_id));
CREATE TABLE product (product_id INTEGER PRIMARY KEY AUTOINCREMENT, supplier_id INTEGER, category_id INTEGER, sku TEXT, name TEXT, origin TEXT, roast_level TEXT, price REAL, product_cost REAL, stock_quantity INTEGER, image TEXT, create_date TEXT, FOREIGN KEY (supplier_id) REFERENCES supplier(supplier_id), FOREIGN KEY (category_id) REFERENCES product_category(category_id));
CREATE TABLE plan_product (plan_product_id INTEGER PRIMARY KEY AUTOINCREMENT, subscription_plan_id INTEGER, product_id INTEGER, FOREIGN KEY (subscription_plan_id) REFERENCES subscription_plan(subscription_plan_id), FOREIGN KEY (product_id) REFERENCES product(product_id));
CREATE TABLE cancellation (cancellation_id INTEGER PRIMARY KEY AUTOINCREMENT, subscription_id INTEGER, cancel_reason_id INTEGER, cancel_date TEXT, comment TEXT, FOREIGN KEY (subscription_id) REFERENCES subscription(subscription_id), FOREIGN KEY (cancel_reason_id) REFERENCES cancel_reason(reason_id));
CREATE TABLE review (review_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER, product_id INTEGER, order_id INTEGER, rating INTEGER, comment TEXT, review_date TEXT, is_verified_purchase INTEGER, FOREIGN KEY (customer_id) REFERENCES customer(customer_id), FOREIGN KEY (product_id) REFERENCES product(product_id), FOREIGN KEY (order_id) REFERENCES "order"(order_id));
CREATE TABLE "order" (order_id INTEGER PRIMARY KEY AUTOINCREMENT, subscription_id INTEGER, customer_id INTEGER, shipping_address_id INTEGER, billing_address_id INTEGER, order_type TEXT, order_date TEXT, order_total REAL, order_status TEXT, FOREIGN KEY (subscription_id) REFERENCES subscription(subscription_id), FOREIGN KEY (customer_id) REFERENCES customer(customer_id), FOREIGN KEY (shipping_address_id) REFERENCES address(address_id), FOREIGN KEY (billing_address_id) REFERENCES address(address_id));
CREATE TABLE order_item (order_item_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER, product_id INTEGER, quantity INTEGER, unit_price REAL, line_total REAL, FOREIGN KEY (order_id) REFERENCES "order"(order_id), FOREIGN KEY (product_id) REFERENCES product(product_id));
CREATE TABLE payment (payment_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER, amount REAL, status TEXT, stripe_trans_id TEXT, payment_date TEXT, payment_method TEXT, FOREIGN KEY (order_id) REFERENCES "order"(order_id));
CREATE TABLE shipment (shipment_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER, tracking_number TEXT, carrier TEXT, ship_date TEXT, status TEXT, FOREIGN KEY (order_id) REFERENCES "order"(order_id));
CREATE TABLE refund (refund_id INTEGER PRIMARY KEY AUTOINCREMENT, payment_id INTEGER, amount REAL, refund_date TEXT, reason TEXT, FOREIGN KEY (payment_id) REFERENCES payment(payment_id));
''')

In [ ]:
# 2. Generating synthetic data

supplier_origins = {1: 'Italy', 2: 'Germany', 3: 'UK', 4: 'Brazil', 5: 'Colombia', 6: 'Rwanda', 7: 'Belgium', 8: 'Honduras', 9: 'Ethiopia', 10: 'Vietnam', 11: 'Indonesia', 12: 'Peru'}
suppliers = [{'supplier_id': 81000000 + i, 'company_name': fake.company(), 'contact_name': fake.name(), 'email': fake.company_email(), 'country': supplier_origins.get(i, fake.country())} for i in range(1, 13)]

categories = [{'category_id': 10000001, 'category_name': 'Machine'}, {'category_id': 10000002, 'category_name': 'Coffee'}]

base_products = [
    # Machines
    {'product_id': 11000001, 'supplier_id': 81000001, 'category_id': 10000001, 'sku': 'BC-11000001', 'name': 'Maestri House Espresso', 'origin': None, 'roast_level': None, 'retail_price': 119.0, 'product_cost': 71.0},
    {'product_id': 11000002, 'supplier_id': 81000002, 'category_id': 10000001, 'sku': 'BC-11000002', 'name': 'DeLonghi Dedica', 'origin': None, 'roast_level': None, 'retail_price': 249.0, 'product_cost': 149.0},
    {'product_id': 11000003, 'supplier_id': 81000003, 'category_id': 10000001, 'sku': 'BC-11000003', 'name': 'Sage Bambino', 'origin': None, 'roast_level': None, 'retail_price': 329.0, 'product_cost': 197.4},
    # Coffee Beans
    {'product_id': 11000004, 'supplier_id': 81000004, 'category_id': 10000002, 'sku': 'BC-11000004', 'name': 'House Blend', 'origin': supplier_origins[4], 'roast_level': 'Dark', 'retail_price': 11.5, 'product_cost': 4.0},
    {'product_id': 11000005, 'supplier_id': 81000005, 'category_id': 10000002, 'sku': 'BC-11000005', 'name': 'Bourbon Cream Espresso', 'origin': supplier_origins[5], 'roast_level': 'Dark', 'retail_price': 12.5, 'product_cost': 4.5},
    {'product_id': 11000006, 'supplier_id': 81000006, 'category_id': 10000002, 'sku': 'BC-11000006', 'name': 'Kigembe Women\'s', 'origin': supplier_origins[6], 'roast_level': 'Light', 'retail_price': 13.5, 'product_cost': 4.75},
    {'product_id': 11000007, 'supplier_id': 81000007, 'category_id': 10000002, 'sku': 'BC-11000007', 'name': 'Tutti Frutti Espresso', 'origin': supplier_origins[7], 'roast_level': 'Medium', 'retail_price': 14.0, 'product_cost': 5.2},
    {'product_id': 11000008, 'supplier_id': 81000008, 'category_id': 10000002, 'sku': 'BC-11000008', 'name': 'Los Ángeles Papayo', 'origin': supplier_origins[8], 'roast_level': 'Light', 'retail_price': 17.0, 'product_cost': 7.0},
    {'product_id': 11000009, 'supplier_id': 81000009, 'category_id': 10000002, 'sku': 'BC-11000009', 'name': 'Mi Regalo', 'origin': supplier_origins[9], 'roast_level': 'Medium', 'retail_price': 17.0, 'product_cost': 7.2},
    {'product_id': 11000010, 'supplier_id': 81000010, 'category_id': 10000002, 'sku': 'BC-11000010', 'name': 'Robusta Dark Roast', 'origin': supplier_origins[10], 'roast_level': 'Dark', 'retail_price': 9.0, 'product_cost': 2.0},
    {'product_id': 11000011, 'supplier_id': 81000011, 'category_id': 10000002, 'sku': 'BC-11000011', 'name': 'Sumatra Earth', 'origin': supplier_origins[11], 'roast_level': 'Medium', 'retail_price': 10.0, 'product_cost': 2.5},
    {'product_id': 11000012, 'supplier_id': 81000012, 'category_id': 10000002, 'sku': 'BC-11000012', 'name': 'Andean Bitter', 'origin': supplier_origins[12], 'roast_level': 'Dark', 'retail_price': 9.5, 'product_cost': 2.2}
]

products = []
for p in base_products:
    p['stock_quantity'] = random.randint(10, 500)
    p['image'] = f"photo_{random.randint(1000, 9999)}.jpg"
    p['create_date'] = start_date.strftime("%Y-%m-%d")
    products.append(p)

# Problematic products are the machines from Supplier 1 and 2
problematic_product_ids = [11000001, 11000002]

subscription_plans = [
    {'subscription_plan_id': 52000001, 'plan_name': 'Classic', 'price': 8.95, 'description': 'Choose from 3 signature blends.'},
    {'subscription_plan_id': 52000002, 'plan_name': 'Premium', 'price': 9.95, 'description': 'Everything in Tier 1, plus 3 moderate single-origin blends.'},
    {'subscription_plan_id': 52000003, 'plan_name': 'Expert', 'price': 12.95, 'description': 'Everything in Tier 2, plus 3 premium coffees.'}
]

# DYNAMIC SUBSCRIPTION PLAN ASSIGNMENT BY PRICE
coffee_beans = sorted([p for p in products if p['category_id'] == 10000002], key=lambda x: x['retail_price'])
cheap_3_ids = [p['product_id'] for p in coffee_beans[0:3]]
moderate_3_ids = [p['product_id'] for p in coffee_beans[3:6]]
expensive_3_ids = [p['product_id'] for p in coffee_beans[6:9]]

plan_products = []
pp_id = 12000001

for pid in cheap_3_ids: # Tier 1 gets cheap 3
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000001, 'product_id': pid}); pp_id += 1
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000002, 'product_id': pid}); pp_id += 1
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000003, 'product_id': pid}); pp_id += 1
for pid in moderate_3_ids: # Tier 2 adds moderate 3
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000002, 'product_id': pid}); pp_id += 1
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000003, 'product_id': pid}); pp_id += 1
for pid in expensive_3_ids: # Tier 3 adds expensive 3
    plan_products.append({'plan_product_id': pp_id, 'subscription_plan_id': 52000003, 'product_id': pid}); pp_id += 1

# Cancel Reason
expanded_cancel_reasons = ['Too expensive', 'No longer drinking coffee', 'Found a different roaster', 'Moving away', 'Quality wasn\'t as expected', 'Overstocked / Need a break', 'Delivery issues', 'Financial constraints', 'Poor customer service', 'Wanted different coffee varieties', 'Taste was terrible']
cancel_reasons = [{'reason_id': 54000001 + i, 'reason_label': r} for i, r in enumerate(expanded_cancel_reasons)]

cancel_comments_map = {
    54000001: ["Just can't afford it right now.", "Price increase was too much."], # Too expensive
    54000002: ["Switched to tea.", "Doctor told me to stop caffeine."], # No longer drinking coffee
    54000003: ["Trying a local roaster.", "Found a cheaper alternative."], # Found a different roaster
    54000004: ["Moving out of the country.", "Moving house."], # Moving away
    54000005: ["The last few batches were too bitter.", "Quality seems to have dropped."], # Quality wasn't as expected
    54000006: ["I have way too much coffee built up!", "Still have 3 bags left."], # Overstocked
    54000007: ["My packages keep arriving late.", "Box was damaged last time."], # Delivery issues
    54000008: ["Lost my job, need to save money.", "Budget is tight this month."], # Financial constraints
    54000009: ["Never got a reply to my email.", "Support was unhelpful."], # Poor customer service
    54000010: ["Want to try decaf options you don't have.", "Need more variety."], # Wanted different varieties
    54000011: ["Could not stomach the taste.", "Really disliked the flavor profile."] # Taste was terrible
}

customers = []
domains = ['@gmail.com', '@yahoo.com', '@outlook.com', '@hotmail.com']
generated_emails = set() # NEW: Keeps track to guarantee 100% uniqueness

for i in range(1, 532):
    gender = random.choice(['Male', 'Female'])
    f_name = fake.first_name_male() if gender == 'Male' else fake.first_name_female()
    l_name = fake.last_name()

    # Email
    while True:
        format_choice = random.choice(['dot_number', 'underscore_year', 'initial_last', 'merged'])
        domain = random.choice(domains)

        if format_choice == 'dot_number':
            email = f"{f_name.lower()}.{l_name.lower()}{random.randint(1, 999)}{domain}"
        elif format_choice == 'underscore_year':
            email = f"{f_name.lower()}_{l_name.lower()}{random.randint(1960, 2005)}{domain}"
        elif format_choice == 'initial_last':
            email = f"{f_name.lower()[0]}{l_name.lower()}{random.randint(100, 9999)}{domain}"
        else:
            email = f"{f_name.lower()}{l_name.lower()}{domain}"

        # Check if this exact email has already been used
        if email not in generated_emails:
            generated_emails.add(email)
            break # Success! Break out of the loop and use this email

    phone = f"07{random.randint(100000000, 999999999)}"

    customers.append({
        'customer_id': 71000000 + i, 'first_name': f_name, 'last_name': l_name,
        'email': email, 'phone': phone, 'gender': gender,
        'date_of_birth': fake.date_of_birth(minimum_age=18, maximum_age=80).strftime("%Y-%m-%d"),
        'join_date': ""
    })

customer_ids = [c['customer_id'] for c in customers]

customer_ids = [c['customer_id'] for c in customers]

# Address

city_postcode_map = {
    'London': ['E', 'EC', 'N', 'NW', 'SE', 'SW', 'W', 'WC'],
    'Birmingham': ['B'], 'Manchester': ['M'], 'Glasgow': ['G'],
    'Newcastle': ['NE'], 'Sheffield': ['S'], 'Leeds': ['LS'],
    'Bristol': ['BS'], 'Liverpool': ['L'], 'Edinburgh': ['EH'],
    'Leicester': ['LE'], 'Coventry': ['CV'], 'Cardiff': ['CF'],
    'Belfast': ['BT'], 'Nottingham': ['NG'], 'Southampton': ['SO'],
    'Reading': ['RG'], 'Brighton': ['BN'], 'Hull': ['HU'],
    'Plymouth': ['PL'], 'Aberdeen': ['AB'], 'Swansea': ['SA'],
    'Oxford': ['OX'], 'Cambridge': ['CB'], 'York': ['YO']
}

uk_cities = list(city_postcode_map.keys())

# Helper function to generate a mathematically valid UK postcode format
def generate_realistic_postcode(city):
    prefix = random.choice(city_postcode_map[city])
    district = random.randint(1, 20)
    sector = random.randint(1, 9)
    # UK postcodes don't use C, I, K, M, O, V in the last two letters
    letters = "".join(random.choices("ABDDEFGHJLNPQRSTUWXYZ", k=2))
    return f"{prefix}{district} {sector}{letters}"

customer_address_map = {c_id: [] for c_id in customer_ids}
addresses = []
a_id = 21000001

# Exactly 531 Default Addresses (1 per customer)
for c_id in customer_ids:
    chosen_city = random.choice(uk_cities)
    matched_postcode = generate_realistic_postcode(chosen_city)

    addresses.append({'address_id': a_id, 'customer_id': c_id, 'address_type': random.choice(['Billing', 'Home']), 'street_line1': fake.street_address(), 'street_line2': fake.street_name() if random.random() > 0.7 else "", 'city': chosen_city, 'postcode': matched_postcode, 'country_code': 'GB', 'is_default': 1})
    customer_address_map[c_id].append(a_id)
    a_id += 1

# Exactly 231 Non-Default Addresses (making 762 total)
for _ in range(231):
    c_id = random.choice(customer_ids)
    chosen_city = random.choice(uk_cities)
    matched_postcode = generate_realistic_postcode(chosen_city)

    addresses.append({'address_id': a_id, 'customer_id': c_id, 'address_type': random.choice(['Work', 'Holiday']), 'street_line1': fake.street_address(), 'street_line2': fake.street_name() if random.random() > 0.7 else "", 'city': chosen_city, 'postcode': matched_postcode, 'country_code': 'GB', 'is_default': 0})
    customer_address_map[c_id].append(a_id)
    a_id += 1

subscriptions, cancellations = [], []
customer_sub_map = {}
cancel_info_map = {}
cancel_id = 53000001

# Cancellation

# Early Cancel: 1m: 91, 2m: 68, 3m: 33, 4m: 19, 5m: 16 (Total: 227)
early_cancels = ([1]*91) + ([2]*68) + ([3]*33) + ([4]*19) + ([5]*16)

# Long Term (6+ months): 304 total. 70% active = 213. 30% cancelled = 91.
long_term_active = 213
long_term_cancel = 91

random.shuffle(customer_ids)
customer_cohorts = []
for _ in range(len(early_cancels)): customer_cohorts.append(('early_cancel', early_cancels.pop(0)))
for _ in range(long_term_cancel): customer_cohorts.append(('long_cancel', random.randint(6, 11)))
for _ in range(long_term_active): customer_cohorts.append(('active', random.randint(6, 11)))

for i, c_id in enumerate(customer_ids):
    sub_id = 51000000 + i + 1
    cohort_type, duration_months = customer_cohorts[i]
    start_dt = end_date - relativedelta(months=duration_months) - relativedelta(days=random.randint(1, 15))

    for c in customers:
        if c['customer_id'] == c_id:
            c['join_date'] = start_dt.strftime("%Y-%m-%d %H:%M:%S")
            break

    if cohort_type in ['early_cancel', 'long_cancel']:
        sub_plan_id = random.choices([52000001, 52000002, 52000003], weights=[20, 20, 60])[0]
    else:
        sub_plan_id = random.choices([52000001, 52000002, 52000003], weights=[45, 45, 10])[0]

    status = 'Active' if cohort_type == 'active' else 'Cancelled'
    next_bill = start_dt + timedelta(days=30)

    subscriptions.append({'subscription_id': sub_id, 'customer_id': c_id, 'subscription_plan_id': sub_plan_id, 'start_date': start_dt.strftime("%Y-%m-%d"), 'next_billing_date': next_bill.strftime("%Y-%m-%d"), 'status': status, 'auto_renew': 1 if status == 'Active' else 0})
    customer_sub_map[c_id] = sub_id

    if status == 'Cancelled':
        cancel_dt = start_dt + relativedelta(months=duration_months)
        reason_id = random.randint(54000001, 54000011)

        # THE FIX: This now looks up the exact comments linked to the reason chosen above!
        c_comment = random.choice(cancel_comments_map[reason_id]) if random.random() > 0.3 else ""

        cancel_record = {'cancellation_id': cancel_id, 'subscription_id': sub_id, 'cancel_reason_id': reason_id, 'cancel_date': cancel_dt.strftime("%Y-%m-%d"), 'comment': c_comment}
        cancellations.append(cancel_record)
        cancel_info_map[sub_id] = cancel_record
        cancel_id += 1

orders, order_items, payments, shipments, refunds = [], [], [], [], []
oi_id, pmt_id, ship_id, ref_id = 32000001, 41000001, 33000001, 42000001

order_types_pool = (['AD_HOC_HARDWARE'] * 2574) + (['AD_HOC_COFFEE'] * 1257) + (['RECURRING'] * 4411)
random.shuffle(order_types_pool)

plan_to_products_map = {}
for pp in plan_products:
    plan_to_products_map.setdefault(pp['subscription_plan_id'], []).append(pp['product_id'])
product_price_map = {p['product_id']: p['retail_price'] for p in products}
plan_price_map = {p['subscription_plan_id']: p['price'] for p in subscription_plans}

refund_indices = set(random.sample(range(8242), 2637))
refund_reasons = ['Item arrived damaged', 'Package lost in transit', 'Wrong item received', 'Coffee was stale / Quality issue', 'Accidental duplicate charge', 'Cancelled before fulfillment']

print(" -> Generating Orders & Transactions...")
for i, o_type in enumerate(order_types_pool):
    o_id = 31000000 + i + 1

    chosen_sub = random.choice(subscriptions)
    linked_sub_id = chosen_sub['subscription_id']
    c_id = chosen_sub['customer_id']
    sub_plan_id = chosen_sub['subscription_plan_id']

    ship_address_id = customer_address_map[c_id][0] # Uses their default address
    bill_address_id = ship_address_id

    sub_start_dt = datetime.strptime(chosen_sub['start_date'], "%Y-%m-%d")
    if chosen_sub['status'] == 'Cancelled':
        sub_end_dt = datetime.strptime(cancel_info_map[linked_sub_id]['cancel_date'], "%Y-%m-%d")
    else:
        sub_end_dt = end_date
    if sub_start_dt >= sub_end_dt: sub_start_dt = sub_end_dt - timedelta(days=1)

# Date Sequencing

    # 1. Order Time
    order_dt = fake.date_time_between(start_date=sub_start_dt, end_date=sub_end_dt)
    order_datetime_str = order_dt.strftime("%Y-%m-%d %H:%M:%S")

    # 2. Payment Time (1 to 48 hours AFTER order)
    payment_dt = order_dt + timedelta(hours=random.randint(1, 48), minutes=random.randint(1, 59))
    payment_date_str = payment_dt.strftime("%Y-%m-%d %H:%M:%S") # Added time!

    # 3. Shipment Time (1 to 3 days AFTER payment)
    ship_dt = payment_dt + timedelta(days=random.randint(1, 3), hours=random.randint(1, 12))
    ship_date_str = ship_dt.strftime("%Y-%m-%d %H:%M:%S") # Added time!

    final_order_type = o_type
    if o_type == 'RECURRING' and random.random() < 0.1:
        final_order_type = 'RECURRING_AND_AD_HOC'

    current_order_items = []
    order_total = 0.0

    if final_order_type in ['RECURRING', 'RECURRING_AND_AD_HOC']:
        chosen_prod_id = random.choice(plan_to_products_map[sub_plan_id])
        qty = 1
        unit_price = plan_price_map[sub_plan_id]
        line_tot = qty * unit_price
        current_order_items.append({'order_item_id': oi_id, 'order_id': o_id, 'product_id': chosen_prod_id, 'quantity': qty, 'unit_price': unit_price, 'line_total': line_tot})
        oi_id += 1
        order_total += line_tot

    if final_order_type in ['AD_HOC_HARDWARE', 'AD_HOC_COFFEE', 'RECURRING_AND_AD_HOC']:
        num_items = random.randint(1, 2)
        if final_order_type == 'AD_HOC_HARDWARE':
            possible_prods = [p['product_id'] for p in products if p['category_id'] == 10000001]
        else:
            possible_prods = [p['product_id'] for p in products if p['category_id'] == 10000002]

        chosen_prods = random.sample(possible_prods, min(num_items, len(possible_prods)))

        for chosen_prod_id in chosen_prods:
            qty = random.randint(1, 2)
            unit_price = product_price_map[chosen_prod_id]
            line_tot = qty * unit_price
            current_order_items.append({'order_item_id': oi_id, 'order_id': o_id, 'product_id': chosen_prod_id, 'quantity': qty, 'unit_price': unit_price, 'line_total': line_tot})
            oi_id += 1
            order_total += line_tot

    order_items.extend(current_order_items)

    days_old = (end_date.date() - order_dt.date()).days
    is_refund = i in refund_indices

    if is_refund:
        scenario = 'Refunded_Post_Delivery'
    else:
        if days_old <= 2:
            scenario = random.choices(['Pending', 'Paid', 'Dispatched', 'Delivered', 'Failed', 'Cancelled_Pre_Dispatch'], weights=[10, 10, 20, 50, 5, 5])[0]
        else:
            scenario = random.choices(['Dispatched', 'Delivered', 'Failed', 'Cancelled_Pre_Dispatch'], weights=[15, 75, 5, 5])[0]

    ship_status = None
    chosen_carrier = None

    if scenario == 'Pending': order_status, pmt_status = 'Pending', 'Pending'
    elif scenario == 'Paid': order_status, pmt_status = 'Paid', 'Completed'
    elif scenario == 'Dispatched': order_status, pmt_status, ship_status = 'Dispatched', 'Completed', 'In transit'
    elif scenario == 'Delivered': order_status, pmt_status, ship_status = 'Delivered', 'Completed', 'Delivered'
    elif scenario == 'Failed': order_status, pmt_status = 'Cancelled', 'Failed'
    elif scenario == 'Cancelled_Pre_Dispatch': order_status, pmt_status, ship_status = 'Cancelled', 'Refunded', 'Cancelled'
    elif scenario == 'Refunded_Post_Delivery': order_status, pmt_status, ship_status = 'Refunded', 'Refunded', 'Cancelled'

    if ship_status is not None:
        chosen_carrier = random.choice(['Royal Mail', 'DPD', 'Evri', 'DHL'])

    orders.append({'order_id': o_id, 'subscription_id': linked_sub_id, 'customer_id': c_id, 'shipping_address_id': ship_address_id, 'billing_address_id': bill_address_id, 'order_type': final_order_type, 'order_date': order_datetime_str, 'order_total': round(order_total, 2), 'order_status': order_status})
    payments.append({'payment_id': pmt_id, 'order_id': o_id, 'amount': round(order_total, 2), 'status': pmt_status, 'stripe_trans_id': fake.uuid4(), 'payment_date': payment_date_str, 'payment_method': random.choice(['Credit', 'Debit'])})

    if ship_status is not None:
        shipments.append({'shipment_id': ship_id, 'order_id': o_id, 'tracking_number': fake.bothify(text='TRK##########'), 'carrier': chosen_carrier, 'ship_date': ship_date_str, 'status': ship_status})
        ship_id += 1

# Refund reasons
    machine_refund_reasons = [
        'Item arrived physically damaged',
        'Package lost in transit',
        'Wrong machine model received',
        'Defective unit / Stopped working',
        'Missing parts in box',
        'Cancelled before fulfillment'
    ]

    coffee_refund_reasons = [
        'Bag ripped / Beans spilled',
        'Package lost in transit',
        'Wrong roast received',
        'Coffee was stale / Quality issue',
        'Cancelled before fulfillment'
    ]

    if pmt_status == 'Refunded':
        # 1. Generate a valid refund date (4 to 10 days after payment)
        r_dt = payment_dt + timedelta(days=random.randint(4, 10), hours=random.randint(1, 8))

        # 2. FOOLPROOF CHECK: Look inside the actual shopping cart for this specific order
        has_machine = any(item['product_id'] in [11000001, 11000002, 11000003] for item in current_order_items)

        # 3. Assign the strictly correct reason pool
        if has_machine:
            chosen_reason = random.choice([
                'Item arrived physically damaged',
                'Package lost in transit',
                'Wrong machine model received',
                'Defective unit / Stopped working',
                'Missing parts in box',
                'Cancelled before fulfillment'
            ])
        else:
            chosen_reason = random.choice([
                'Bag ripped / Beans spilled',
                'Package lost in transit',
                'Wrong roast received',
                'Coffee was stale / Quality issue',
                'Cancelled before fulfillment'
            ])

        refunds.append({
            'refund_id': ref_id,
            'payment_id': pmt_id,
            'amount': round(order_total, 2),
            'refund_date': r_dt.strftime("%Y-%m-%d"),
            'reason': chosen_reason
        })
        ref_id += 1

    pmt_id += 1

# Review Comments

# Machine Specific Comments
machine_pos = ["Amazing espresso maker!", "Great machine.", "Perfect extraction every time.", "Heats up incredibly fast!", "Best hardware purchase for my home.", "Highly recommend this machine.", "Beautiful design.", "Makes mornings better.", "Outstanding build quality.", "Easy to use.", "Fantastic extraction.", "A game changer for my kitchen.", "Premium feel.", "Sturdy and reliable.", "Incredible steam power."]
machine_neu = ["It's okay for the price.", "Nothing special hardware-wise.", "Standard espresso maker.", "A bit noisy.", "I've had better machines.", "Average build.", "A little overpriced.", "Does the job fine.", "Function is just okay.", "A bit weak steam wand.", "Noisy but functions.", "Meets basic expectations.", "Requires dialing in.", "Water tank is slightly small.", "Hard to clean."]
machine_neg = ["Terrible machine.", "Broke easily.", "Water leaked everywhere.", "Stopped working.", "Grinder clogged immediately.", "Awful quality control.", "Completely defective.", "Overpriced garbage.", "Cheap plastic parts.", "Ruined my kitchen counter.", "Do not waste your money.", "Portafilter doesn't fit.", "Steam wand has no power.", "Incredibly loud.", "Very disappointed in the build.", "Defective unit."]

# Coffee Specific Comments
coffee_pos = ["Amazing!", "Great taste.", "Perfectly roasted.", "Diverse coffee variety.", "My go-to morning coffee.", "Smooth and flavorful.", "Outstanding bean quality.", "Rich taste, no bitterness.", "Highly recommend.", "Delivered fresh.", "Beautiful packaging.", "Delightful blend.", "Consistently amazing.", "Best coffee in a long time.", "Exceeded expectations.", "Love the rich crema.", "Incredible aroma.", "Tastes like a premium cafe.", "Smooth finish.", "Bright and fruity.", "Perfect balance of acidity."]
coffee_neu = ["It's okay.", "Nothing special.", "Standard everyday coffee.", "A bit too acidic.", "I've had better beans.", "Average taste.", "A little overpriced for beans.", "Flavor is just okay.", "A bit weak.", "Fine for a quick cup.", "Meets basic flavor expectations.", "Wouldn't rush to buy again.", "Middle of the road blend.", "Okay quality.", "Missing complex notes.", "Acceptable.", "Fairly standard dark roast."]
coffee_neg = ["Terrible.", "Watery flavor.", "Old roast date.", "Tastes burnt.", "Way too sour.", "Tastes like dirt.", "Beans were dry.", "No crema at all.", "Ruined my morning.", "Do not waste your money on this blend.", "Under-developed.", "Worst coffee I've ever bought.", "Very disappointed.", "Wrong bean sent.", "Bitter and stale.", "Terrible quality.", "Stale beans."]

print(" -> Generating Reviews...")
reviews = []

# Map order_id directly to customer_id so the script runs instantly
order_to_customer = {o['order_id']: o['customer_id'] for o in orders}

# UNIQUE CONSTRAINT: Build a dictionary using (c_id, p_id) as the key.
# If a customer bought the same coffee 12 times, it just overwrites the key and only leaves 1 record!
unique_customer_product_pairs = {}
for item in order_items:
    o_id = item['order_id']
    p_id = item['product_id']
    c_id = order_to_customer[o_id]

    unique_customer_product_pairs[(c_id, p_id)] = o_id

# Now we sample 3,000 reviews from our perfectly de-duplicated list
review_candidates = random.sample(
    list(unique_customer_product_pairs.items()),
    min(3000, len(unique_customer_product_pairs))
)

review_id = 61000001

for (c_id, p_id), o_id in review_candidates:
    # Check if the product is a Machine (IDs 1-3)
    is_machine = p_id in [11000001, 11000002, 11000003]

    # Negative weight increased drastically for machines from Problematic Suppliers 1 and 2
    if p_id in problematic_product_ids:
        sentiment = random.choices(['positive', 'neutral', 'negative'], weights=[5, 5, 90])[0]
    else:
        # Strict exact weighting for normal products: 42% Pos, 35% Neg, 23% Neu
        sentiment = random.choices(['positive', 'neutral', 'negative'], weights=[42, 23, 35])[0]

    if sentiment == 'positive':
        comment = random.choice(machine_pos) if is_machine else random.choice(coffee_pos)
        rating = random.choice([4, 5])
    elif sentiment == 'neutral':
        comment = random.choice(machine_neu) if is_machine else random.choice(coffee_neu)
        rating = 3
    else:
        comment = random.choice(machine_neg) if is_machine else random.choice(coffee_neg)
        rating = random.choice([1, 2])

    reviews.append({
        'review_id': review_id, 'customer_id': c_id, 'product_id': p_id, 'order_id': o_id, 'rating': rating,
        'comment': comment, 'review_date': fake.date_time_between(start_date='-6m', end_date=end_date).strftime("%Y-%m-%d"),
        'is_verified_purchase': 1
    })
    review_id += 1


In [ ]:
# 4. Insert into SQLITE Database
print("Step 3: Importing data into SQLite database (Optimized)...")
def nullify(val): return None if val == "" else val

cursor.executemany('INSERT INTO supplier (supplier_id, company_name, contact_person, email, country_of_origin) VALUES (?, ?, ?, ?, ?)', [(r['supplier_id'], r['company_name'], r['contact_name'], r['email'], r['country']) for r in suppliers])
cursor.executemany('INSERT INTO product_category (category_id, category_name) VALUES (?, ?)', [(r['category_id'], r['category_name']) for r in categories])
cursor.executemany('INSERT INTO product (product_id, supplier_id, category_id, sku, name, origin, roast_level, price, product_cost, stock_quantity, image, create_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)', [(r['product_id'], r['supplier_id'], r['category_id'], r['sku'], r['name'], r['origin'], r['roast_level'], r['retail_price'], r['product_cost'], r['stock_quantity'], r['image'], r['create_date']) for r in products])
cursor.executemany('INSERT INTO subscription_plan (subscription_plan_id, subscription_plan_name, price, description) VALUES (?, ?, ?, ?)', [(r['subscription_plan_id'], r['plan_name'], r['price'], r['description']) for r in subscription_plans])
cursor.executemany('INSERT INTO plan_product (plan_product_id, subscription_plan_id, product_id) VALUES (?, ?, ?)', [(r['plan_product_id'], r['subscription_plan_id'], r['product_id']) for r in plan_products])
cursor.executemany('INSERT INTO cancel_reason (reason_id, reason_label) VALUES (?, ?)', [(r['reason_id'], r['reason_label']) for r in cancel_reasons])
cursor.executemany('INSERT INTO customer (customer_id, first_name, last_name, email, phone, gender, date_of_birth, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)', [(r['customer_id'], r['first_name'], r['last_name'], r['email'], r['phone'], r['gender'], r['date_of_birth'], r['join_date']) for r in customers])
cursor.executemany('INSERT INTO address (address_id, customer_id, address_type, street_line1, street_line2, postcode, city, country_code, is_default) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', [(r['address_id'], r['customer_id'], r['address_type'], r['street_line1'], r['street_line2'], r['postcode'], r['city'], r['country_code'], r['is_default']) for r in addresses])
cursor.executemany('INSERT INTO subscription (subscription_id, customer_id, subscription_plan_id, start_date, next_billing_date, status, auto_renew) VALUES (?, ?, ?, ?, ?, ?, ?)', [(r['subscription_id'], r['customer_id'], r['subscription_plan_id'], r['start_date'], r['next_billing_date'], r['status'], r['auto_renew']) for r in subscriptions])
cursor.executemany('INSERT INTO cancellation (cancellation_id, subscription_id, cancel_reason_id, cancel_date, comment) VALUES (?, ?, ?, ?, ?)', [(r['cancellation_id'], r['subscription_id'], r['cancel_reason_id'], r['cancel_date'], r['comment']) for r in cancellations])
cursor.executemany('INSERT INTO "order" (order_id, subscription_id, customer_id, shipping_address_id, billing_address_id, order_type, order_date, order_total, order_status) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', [(r['order_id'], r.get('subscription_id'), r['customer_id'], r.get('shipping_address_id'), r.get('billing_address_id'), r['order_type'], r['order_date'], r['order_total'], r['order_status']) for r in orders])
cursor.executemany('INSERT INTO order_item (order_item_id, order_id, product_id, quantity, unit_price, line_total) VALUES (?, ?, ?, ?, ?, ?)', [(r['order_item_id'], r['order_id'], r['product_id'], r['quantity'], r['unit_price'], r['line_total']) for r in order_items])
cursor.executemany('INSERT INTO payment (payment_id, order_id, amount, status, stripe_trans_id, payment_date, payment_method) VALUES (?, ?, ?, ?, ?, ?, ?)', [(r['payment_id'], r['order_id'], r['amount'], r['status'], r['stripe_trans_id'], r['payment_date'], r['payment_method']) for r in payments])
cursor.executemany('INSERT INTO shipment (shipment_id, order_id, tracking_number, carrier, ship_date, status) VALUES (?, ?, ?, ?, ?, ?)', [(r['shipment_id'], r['order_id'], r['tracking_number'], r['carrier'], r['ship_date'], r['status']) for r in shipments])
cursor.executemany('INSERT INTO refund (refund_id, payment_id, amount, refund_date, reason) VALUES (?, ?, ?, ?, ?)', [(r['refund_id'], r['payment_id'], r['amount'], r['refund_date'], r['reason']) for r in refunds])
cursor.executemany('INSERT INTO review (review_id, customer_id, product_id, order_id, rating, comment, review_date, is_verified_purchase) VALUES (?, ?, ?, ?, ?, ?, ?, ?)', [(r['review_id'], r['customer_id'], r['product_id'], r['order_id'], r['rating'], r['comment'], r['review_date'], r['is_verified_purchase']) for r in reviews])

conn.commit()

The following cells answer the five business questions using SQL queries run against the `bean_co.db` SQLite database.  
Each section contains: the query, its output table, and a brief interpretation.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('bean_co.db')
print('Connected to bean_co.db ✓')

### Q1 Growth & Retention
#### Q1a Month-over-Month New User Acquisition
Counts new customers per calendar month and uses a **window function** (`SUM … OVER`) to build a running cumulative total — both in a single query pass.

In [ ]:
q1a = """
SELECT
    strftime('%Y-%m', created_at)                              AS month,
    COUNT(*)                                                   AS new_users,
    SUM(COUNT(*)) OVER (ORDER BY strftime('%Y-%m', created_at)) AS cumulative_users
FROM   customer
GROUP  BY month
ORDER  BY month;
"""

df_q1a = pd.read_sql_query(q1a, conn)
display(df_q1a)

#### Q1b · Average Customer Lifetime Value (CLV) by Subscription Plan
CLV = total completed payments per customer. A **subquery** aggregates revenue per customer first; a **LEFT JOIN** then attaches it to plan data so customers with zero orders are still counted.

In [ ]:
q1b = """
SELECT
    sp.subscription_plan_name        AS plan,
    COUNT(DISTINCT c.customer_id)    AS customers,
    ROUND(AVG(rev.total_revenue), 2) AS avg_clv,
    ROUND(MIN(rev.total_revenue), 2) AS min_clv,
    ROUND(MAX(rev.total_revenue), 2) AS max_clv
FROM customer c
JOIN subscription s
    ON c.customer_id = s.customer_id
JOIN subscription_plan sp
    ON s.subscription_plan_id = sp.subscription_plan_id
LEFT JOIN (
    SELECT   o.customer_id,
             SUM(p.amount) AS total_revenue
    FROM     "order" o
    JOIN     payment p ON o.order_id = p.order_id
    WHERE    p.status = 'Completed'
    GROUP BY o.customer_id
) rev ON c.customer_id = rev.customer_id
GROUP BY  sp.subscription_plan_name
ORDER BY  avg_clv DESC;
"""

df_q1b = pd.read_sql_query(q1b, conn)
display(df_q1b)

#### Q2 — Product Intelligence
##### Q2a · Roast Level Popularity by Subscription Plan
Uses a **window function** (`SUM … OVER PARTITION BY plan`) to compute each roast level's percentage share within its plan — avoiding a self-join.

In [ ]:
q2a = """
SELECT
    sp.subscription_plan_name                                        AS plan,
    p.roast_level,
    SUM(oi.quantity)                                                 AS total_units,
    ROUND(SUM(oi.line_total), 2)                                     AS total_revenue,
    ROUND(
        SUM(oi.quantity) * 100.0
        / SUM(SUM(oi.quantity)) OVER (PARTITION BY sp.subscription_plan_name)
    , 1)                                                             AS pct_of_plan
FROM   order_item oi
JOIN   product p         ON oi.product_id             = p.product_id
JOIN   "order" o          ON oi.order_id               = o.order_id
JOIN   subscription s    ON o.customer_id             = s.customer_id
JOIN   subscription_plan sp ON s.subscription_plan_id = sp.subscription_plan_id
WHERE  p.roast_level IS NOT NULL
GROUP  BY sp.subscription_plan_name, p.roast_level
ORDER  BY plan, total_units DESC;
"""

df_q2a = pd.read_sql_query(q2a, conn)
display(df_q2a)

### Q2b · Country of Origin Popularity by Subscription Plan
Same window-function pattern as Q2a, applied to country of origin — enabling direct tier-by-tier comparison for sourcing decisions.

In [ ]:
q2b = """
SELECT
    sp.subscription_plan_name                                        AS plan,
    p.origin,
    SUM(oi.quantity)                                                 AS total_units,
    ROUND(
        SUM(oi.quantity) * 100.0
        / SUM(SUM(oi.quantity)) OVER (PARTITION BY sp.subscription_plan_name)
    , 1)                                                             AS pct_of_plan
FROM   order_item oi
JOIN   product p         ON oi.product_id             = p.product_id
JOIN   "order" o          ON oi.order_id               = o.order_id
JOIN   subscription s    ON o.customer_id             = s.customer_id
JOIN   subscription_plan sp ON s.subscription_plan_id = sp.subscription_plan_id
WHERE  p.origin IS NOT NULL
GROUP  BY sp.subscription_plan_name, p.origin
ORDER  BY plan, total_units DESC;
"""

df_q2b = pd.read_sql_query(q2b, conn)
display(df_q2b)

---
#### Q3  Consumer Behavior: Machine Buyers
##### Q3 · Age Group, Gender & Subscription Tenure of Customers Who Buy Coffee Machines
Uses **CASE-based bucketing** inside a subquery to classify buyer age, then aggregates by gender and age group. Filters on `category_id = 10000001` (Machine) to isolate hardware purchasers without relying on string matching.

In [ ]:
q3 = """
SELECT
    c.gender,
    CASE
        WHEN age < 30 THEN '18-29'
        WHEN age < 40 THEN '30-39'
        WHEN age < 50 THEN '40-49'
        WHEN age < 60 THEN '50-59'
        ELSE               '60+'
    END                              AS age_group,
    COUNT(DISTINCT c.customer_id)    AS machine_buyers,
    ROUND(AVG(age), 1)               AS avg_age,
    ROUND(AVG(tenure_months), 1)     AS avg_tenure_months
FROM (
    SELECT DISTINCT
        o.customer_id,
        CAST((julianday('2026-02-28') - julianday(c.date_of_birth)) / 365.25 AS INT) AS age,
        CAST((julianday(o.order_date) - julianday(s.start_date))  / 30      AS INT) AS tenure_months
    FROM   "order" o
    JOIN   order_item oi ON o.order_id    = oi.order_id
    JOIN   product p     ON oi.product_id = p.product_id
    JOIN   customer c    ON o.customer_id = c.customer_id
    JOIN   subscription s ON o.customer_id = s.customer_id
    WHERE  p.category_id = 10000001
) buyers
JOIN customer c ON buyers.customer_id = c.customer_id
GROUP  BY c.gender, age_group
ORDER  BY age_group, c.gender;
"""

df_q3 = pd.read_sql_query(q3, conn)
display(df_q3)

---
#### Q4  Churn Analysis
##### Q4a · Cancellation Rate by Subscription Stage (First Box, Second Box, etc.)
Maps each cancellation to the delivery cycle it followed using **day-range buckets** (≤35 days per box cycle). A window function computes the percentage share of each stage across all cancellations.

In [ ]:
q4a = """
SELECT
    CASE
        WHEN days_to_cancel <= 35 THEN '1 — Churned after 1st box'
        WHEN days_to_cancel <= 65 THEN '2 — Churned after 2nd box'
        WHEN days_to_cancel <= 95 THEN '3 — Churned after 3rd box'
        ELSE                           '4 — Long-term churn (4+ boxes)'
    END                                                        AS churn_stage,
    COUNT(*)                                                   AS customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)        AS pct
FROM (
    SELECT
        ca.cancellation_id,
        CAST(julianday(ca.cancel_date) - julianday(s.start_date) AS INT) AS days_to_cancel
    FROM   cancellation ca
    JOIN   subscription s ON ca.subscription_id = s.subscription_id
) sub
GROUP  BY churn_stage
ORDER  BY churn_stage;
"""

df_q4a = pd.read_sql_query(q4a, conn)
display(df_q4a)

### Q4b · Churn Reasons — Early vs Late Breakdown
Uses **conditional aggregation** (`SUM + CASE`) to split each cancellation reason into early-churn (≤35 days) and late-churn (>35 days) columns in a single query — no pivot table or self-join needed.

In [ ]:
q4b = """
SELECT
    cr.reason_label                                                         AS churn_reason,
    COUNT(*)                                                                AS total,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM cancellation), 1)        AS pct_of_all,
    SUM(CASE
            WHEN CAST(julianday(ca.cancel_date)
                    - julianday(s.start_date) AS INT) <= 35
            THEN 1 ELSE 0 END)                                              AS early_churn,
    SUM(CASE
            WHEN CAST(julianday(ca.cancel_date)
                    - julianday(s.start_date) AS INT) > 35
            THEN 1 ELSE 0 END)                                              AS late_churn
FROM   cancellation ca
JOIN   cancel_reason cr ON ca.cancel_reason_id = cr.reason_id
JOIN   subscription s   ON ca.subscription_id  = s.subscription_id
GROUP  BY cr.reason_label
ORDER  BY total DESC;
"""

df_q4b = pd.read_sql_query(q4b, conn)
display(df_q4b)

---
#### Q5 Financial Health
##### Q5a · MRR vs Hardware Revenue — Monthly Trend
Uses **conditional aggregation** to split each completed payment into its revenue stream (RECURRING / AD_HOC_HARDWARE / AD_HOC_COFFEE) in a single pass. `NULLIF` prevents divide-by-zero when computing MRR's share of total revenue.

In [ ]:
q5a = """
SELECT
    strftime('%Y-%m', o.order_date)                                          AS month,
    ROUND(SUM(CASE WHEN o.order_type = 'RECURRING'       THEN p.amount ELSE 0 END), 2) AS mrr,
    ROUND(SUM(CASE WHEN o.order_type = 'AD_HOC_HARDWARE' THEN p.amount ELSE 0 END), 2) AS hardware_revenue,
    ROUND(SUM(CASE WHEN o.order_type = 'AD_HOC_COFFEE'   THEN p.amount ELSE 0 END), 2) AS adhoc_coffee,
    ROUND(SUM(p.amount), 2)                                                  AS total_revenue,
    ROUND(
        SUM(CASE WHEN o.order_type = 'RECURRING' THEN p.amount ELSE 0 END)
        * 100.0 / NULLIF(SUM(p.amount), 0)
    , 1)                                                                     AS mrr_pct
FROM   "order" o
JOIN   payment p ON o.order_id = p.order_id
WHERE  p.status = 'Completed'
GROUP  BY month
ORDER  BY month;
"""

df_q5a = pd.read_sql_query(q5a, conn)
display(df_q5a)

### Q5b · Product Gross Profit & Margin Analysis
Computes gross profit inline by joining `product_cost` from the product table. `NULLIF` guards against divide-by-zero in the margin percentage. Results are ordered by gross revenue to surface the most financially significant products first.

In [ ]:
q5b = """
SELECT
    pr.name                                                           AS product,
    pc.category_name                                                  AS category,
    SUM(oi.quantity)                                                  AS units_sold,
    ROUND(SUM(oi.line_total), 2)                                      AS gross_revenue,
    ROUND(SUM(oi.quantity * pr.product_cost), 2)                      AS total_cost,
    ROUND(SUM(oi.line_total) - SUM(oi.quantity * pr.product_cost), 2) AS gross_profit,
    ROUND(
        (SUM(oi.line_total) - SUM(oi.quantity * pr.product_cost))
        * 100.0 / NULLIF(SUM(oi.line_total), 0)
    , 1)                                                              AS margin_pct
FROM   order_item oi
JOIN   product pr         ON oi.product_id  = pr.product_id
JOIN   product_category pc ON pr.category_id = pc.category_id
GROUP  BY pr.name, pc.category_name
ORDER  BY gross_revenue DESC;
"""

df_q5b = pd.read_sql_query(q5b, conn)
display(df_q5b)

In [ ]:
conn.close()
print('Connection closed.')